# Part 3, Topic 3: DPA on Firmware Implementation of AES (HARDWARE)

---
**THIS IS NOT THE COMPLETE TUTORIAL - see file with `(MAIN)` in the name.**

---

First you'll need to select which hardware setup you have. You'll need to select a `SCOPETYPE`, a `PLATFORM`, and a `CRYPTO_TARGET`. `SCOPETYPE` can either be `'OPENADC'` for the CWLite/CW1200 or `'CWNANO'` for the CWNano. `PLATFORM` is the target device, with `'CWLITEARM'`/`'CW308_STM32F3'` being the best supported option, followed by `'CWLITEXMEGA'`/`'CW308_XMEGA'`, then by `'CWNANO'`. `CRYPTO_TARGET` selects the crypto implementation, with `'TINYAES128C'` working on all platforms. An alternative for `'CWLITEXMEGA'` targets is `'AVRCRYPTOLIB'`. For example:

```python
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
CRYPTO_TARGET='TINYAES128C'
SS_VER='SS_VER_1_1'
```

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
CRYPTO_TARGET='TINYAES128C'
SS_VER='SS_VER_1_1'

The following code will build the firmware for the target.

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 7660016                   to 21781610                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 29538471                  to 28823535                 
scope.clock.adc_rate                     changed from 29538471.0                to 28823535.0        

In [3]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j


Building for platform CW308_STM32F3 with CRYPTO_TARGET=TINYAES128C
SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
Blank crypto options, building for AES128
arm-none-eabi-gcc (15:13.2.rel1-2) 13.2.1 20231009
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CW308_STM32F3 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
-en     simpleserial-aes.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
-en     .././hal//stm32f3/stm32f3_hal.c ...
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
.
.
.
Compiling:
.
Compiling:
Compiling:
-en     .././hal//stm32f3/stm32f3_sysmem.c ...
Assembling: .././hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage

For this tutorial, we'll need to capture around 2500 traces:

In [8]:
cw.program_target(scope, prog, "../../../firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM))
ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

ktp = [['2b7e151628aed2a6abf7158809cf4f3c', '1f3b803e3074528768a6277330963d46'], ['2b7e151628aed2a6abf7158809cf4f3c', '703683cf65515cb01bf5fca95afd0c90'], ['2b7e151628aed2a6abf7158809cf4f3c', '5fb1b8cd9ce3c2f36e3edd21e83056fa'], ['2b7e151628aed2a6abf7158809cf4f3c', '50ffa721811c89e0cdf618ee153509f4']]
key = bytearray.fromhex(ktp[0][0])
text = bytearray.fromhex(ktp[0][1])

target.set_key(key)
N = 4
for i in range(N):
    key = bytearray.fromhex(ktp[i][0])
    text = bytearray.fromhex(ktp[i][1])
    scope.arm()
    
    target.simpleserial_write('p', text)
    
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    
    response = target.simpleserial_read('r', 16)
    print(response)
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 6095 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 6095 bytes
bytearray(b'\xcco;\x98\xbe\xa9~\xff=y\x90\xcf\xf3J\xf4\xb7')
bytearray(b'\xa6\xb7_\xf2\xf8p\x1au{iy\x8e&\x84\xd6\x87')
bytearray(b'\xdd\xb6-\xe9\x8b\xcd\xb4G\xdc\x10\xf8L\x9c\xd0\xab\x17')
bytearray(b'J;\x1b\x99\xbe\xb19\x03\xd5%]V\xfc\x948\xa2')


In [ ]:
import matplotlib.pyplot as plt
plt.plot(trace_array[0])
plt.plot(trace_array[1])
plt.show()
